# Pnyx local vLLM server (Colab)

This notebook hosts **`meta-llama/Llama-3.1-8B-Instruct`** behind an OpenAI-compatible
vLLM server on a Colab GPU runtime, and exposes it to the outside world (your Mac,
running the Pnyx harness) via a `cloudflared` quick tunnel. No inbound ports, no
ngrok account, no fixed IP needed.

It backs the **local-model** arms of the P4 experiment grid: condition `B1`
(6x local vLLM homogeneous pool) and the local slice of the mixed pools `C` and
`D_k1` / `D_k3` / `D_k10`. See `pnyx/configs/main/B1.yaml`, `C.yaml`, `D_k1.yaml`,
`D_k3.yaml`, `D_k10.yaml` in the repo — each has a `base_url: REPLACE_WITH_COLAB_URL`
placeholder that Cell 5 below will give you the real value for.

## GPU runtime (required)

`Runtime -> Change runtime type -> T4 GPU` (or better). A **T4 (16GB)** is enough
for Llama-3.1-8B-Instruct in fp16 as long as the context window is capped — this
notebook launches vLLM with `--max-model-len 8192`, which comfortably fits a T4's
KV-cache budget at `--gpu-memory-utilization 0.92`. An **A100** is not required but
is faster to boot (quicker weight download/compile) and gives headroom for larger
batches or a longer context if you change `--max-model-len` later.

## Hugging Face token (gated repo)

`meta-llama/Llama-3.1-8B-Instruct` is a **gated** repo — you must accept its
license before you can download weights:

1. Log into Hugging Face and accept the license at
   https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
2. Create a **read** token at https://huggingface.co/settings/tokens
3. Keep it handy to paste into Cell 3 below (it is only held in the notebook's
   in-memory environment for this session — never echoed, never written to disk).

## How to run

Run the cells **top to bottom, in order**. Cells 4 and 5 launch background
processes and return as soon as the server / tunnel are confirmed up (they poll,
so they can take a few minutes on the very first run while weights download).
Cell 7 (keep-alive) is the only cell meant to block — run it last, and when you
want to end the session just interrupt that cell (stop button), not the whole
runtime, if you want to inspect logs afterwards.


In [ ]:
# Install vLLM + httpx (client for the smoke test) and fetch the cloudflared binary
# (used later to expose :8000 publicly without any inbound networking setup).
#
# Colab's GPU runtime ships CUDA 12.x, but a plain `pip install vllm` can resolve
# a wheel built against CUDA 13, which then fails at import time with
# `ImportError: libcudart.so.13`. Installing via uv with --torch-backend=auto
# makes vLLM auto-select the torch/CUDA build that actually matches this
# runtime's CUDA version, avoiding that mismatch.
!nvidia-smi | head -4

!pip -q install uv
!uv pip install --system -q httpx
!uv pip install --system -q vllm --torch-backend=auto

import os
import urllib.request

CLOUDFLARED_URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
CLOUDFLARED_PATH = "/usr/local/bin/cloudflared"

print("Downloading cloudflared ...")
urllib.request.urlretrieve(CLOUDFLARED_URL, CLOUDFLARED_PATH)
os.chmod(CLOUDFLARED_PATH, 0o755)
print("cloudflared installed at", CLOUDFLARED_PATH)

!cloudflared --version

# Verify the install now, so a bad wheel fails here instead of at serve time.
!python -c "import vllm; print('vllm', vllm.__version__)"

In [ ]:
# Hugging Face token for the gated meta-llama repo.
# Input is hidden (getpass) and the token is only kept in this process's
# environment variables -- it is never printed or written to disk.
import os
from getpass import getpass

hf_token = getpass("Paste your Hugging Face token (hidden input): ")
os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token  # some huggingface_hub/vllm paths check this name
del hf_token
print("HF token set for this session.")


In [ ]:
# Launch `vllm serve` as a background subprocess and wait for it to come up.
#
# vLLM's OpenAI-compatible server already accepts
# `response_format={"type": "json_schema", "json_schema": {...}}` on
# /v1/chat/completions and enforces it via guided decoding out of the box --
# no extra flag is required to turn this on.
import os
import subprocess
import time
import urllib.error
import urllib.request

VLLM_PORT = 8000
VLLM_API_KEY = "pnyx-local"
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
VLLM_LOG_PATH = "/content/vllm_server.log"

vllm_cmd = [
    "vllm", "serve", MODEL_ID,
    "--port", str(VLLM_PORT),
    "--api-key", VLLM_API_KEY,
    "--max-model-len", "8192",
    "--gpu-memory-utilization", "0.92",
]

print("Launching vLLM server (first run can take several minutes while weights")
print("download from the Hub) ...")
print(" ".join(vllm_cmd))

vllm_log = open(VLLM_LOG_PATH, "w")
vllm_proc = subprocess.Popen(
    vllm_cmd,
    stdout=vllm_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)


def wait_for_vllm(proc, port, timeout_s=1800, poll_s=10):
    url = f"http://localhost:{port}/v1/models"
    start = time.time()
    while True:
        if proc.poll() is not None:
            raise RuntimeError(
                f"vLLM process exited early (code {proc.returncode}). "
                f"Check {VLLM_LOG_PATH} for the traceback."
            )
        try:
            with urllib.request.urlopen(url, timeout=5) as resp:
                if resp.status == 200:
                    print(f"\nvLLM server is up on :{port} after {time.time() - start:.0f}s")
                    return
        except (urllib.error.URLError, ConnectionError, TimeoutError, OSError):
            pass
        elapsed = time.time() - start
        if elapsed > timeout_s:
            raise TimeoutError(
                f"vLLM server did not come up within {timeout_s}s. "
                f"Check {VLLM_LOG_PATH} for details."
            )
        print(f"  ... waiting for vLLM ({elapsed:.0f}s elapsed; log: {VLLM_LOG_PATH})")
        time.sleep(poll_s)


wait_for_vllm(vllm_proc, VLLM_PORT)


In [ ]:
# Launch a cloudflared "quick tunnel" that exposes localhost:8000 publicly,
# and parse the printed *.trycloudflare.com URL out of its log.
import re
import subprocess
import time

TUNNEL_LOG_PATH = "/content/cloudflared.log"
_TUNNEL_URL_RE = re.compile(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com")


def start_tunnel(port=VLLM_PORT, log_path=TUNNEL_LOG_PATH):
    log_file = open(log_path, "w")
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
    return proc, log_file


def wait_for_tunnel_url(log_path=TUNNEL_LOG_PATH, timeout_s=120, poll_s=2):
    start = time.time()
    while time.time() - start < timeout_s:
        with open(log_path) as f:
            text = f.read()
        m = _TUNNEL_URL_RE.search(text)
        if m:
            return m.group(0)
        time.sleep(poll_s)
    raise TimeoutError(
        f"cloudflared did not print a public URL within {timeout_s}s; check {log_path}"
    )


cloudflared_proc, cloudflared_log = start_tunnel()
public_url = wait_for_tunnel_url()

print("=" * 72)
print("PUBLIC TUNNEL URL:", public_url)
print("=" * 72)
print(f"Paste this into pnyx/configs/main/*.yaml as: base_url: {public_url}/v1")
print("(targets: B1.yaml, C.yaml, D_k1.yaml, D_k3.yaml, D_k10.yaml)")
print()
print("NOTE: this URL is only valid for this tunnel process. If the tunnel dies")
print("and restarts (see the keep-alive cell) you will get a NEW url and must")
print("update the configs again before rerunning the harness.")


In [ ]:
# Smoke test: hit the PUBLIC url's /v1/chat/completions with a tiny
# structured-output (json_schema) request, exactly like the Pnyx provider will.
import httpx

headers = {
    "Authorization": f"Bearer {VLLM_API_KEY}",
    "Content-Type": "application/json",
}

schema = {
    "type": "object",
    "properties": {
        "answer": {"type": "string"},
        "confidence": {"type": "number", "minimum": 0, "maximum": 1},
    },
    "required": ["answer", "confidence"],
    "additionalProperties": False,
}

payload = {
    "model": MODEL_ID,
    "messages": [
        {"role": "system", "content": "Reply with strict JSON matching the given schema."},
        {"role": "user", "content": "Is the sky blue on a clear day? Answer with a confidence in [0,1]."},
    ],
    "max_tokens": 128,
    "temperature": 0.0,
    "response_format": {
        "type": "json_schema",
        "json_schema": {"name": "smoke_test", "schema": schema, "strict": True},
    },
}

resp = httpx.post(f"{public_url}/v1/chat/completions", headers=headers, json=payload, timeout=60.0)
resp.raise_for_status()
data = resp.json()

print("Parsed reply content:")
print(data["choices"][0]["message"]["content"])
print()
print("Usage:", data.get("usage"))


In [ ]:
# Keep-alive loop. Blocking on purpose -- run this cell last and leave it
# running for the duration of your Colab session. Checks liveness of both the
# vLLM server and the cloudflared tunnel every HEARTBEAT_S seconds, and
# restarts the tunnel (printing the new URL) if it has died. If vLLM itself
# dies, it prints a warning and points you at the log -- re-run Cell 4 to
# bring it back up.
import time
from datetime import datetime, timezone

HEARTBEAT_S = 60


def is_alive(proc):
    return proc is not None and proc.poll() is None


print("Keep-alive loop running. Interrupt this cell (stop button) to end the session.")
print(f"vLLM server:  http://localhost:{VLLM_PORT}")
print(f"Public URL:   {public_url}")

try:
    while True:
        now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        vllm_ok = is_alive(vllm_proc)

        if not vllm_ok:
            print(
                f"[{now}] vLLM server process is DEAD (exit code {vllm_proc.returncode}). "
                f"Check {VLLM_LOG_PATH} and re-run Cell 4 to restart it."
            )
        elif not is_alive(cloudflared_proc):
            print(f"[{now}] cloudflared tunnel is DEAD -- restarting ...")
            cloudflared_proc, cloudflared_log = start_tunnel()
            public_url = wait_for_tunnel_url()
            print(f"[{now}] New public URL: {public_url}  -- UPDATE YOUR CONFIGS.")
        else:
            print(f"[{now}] heartbeat OK -- vLLM up, tunnel up, url={public_url}")

        time.sleep(HEARTBEAT_S)
except KeyboardInterrupt:
    print("Keep-alive loop stopped by user.")
